# 03 — TDCS LoRA SFT on Korean Olympiad TDCS 3K
Train with the five-level TDCS curriculum, then evaluate on `ChuGyouk/OlympiadBench-Math-Ko` using the same symbolic-equivalence scorer as Notebooks 1 and 2.

In [ ]:
REPO_URL = "https://github.com/seungjun-green/Korean-TDCS"
!git clone {REPO_URL} korean-math-tdcs
%cd korean-math-tdcs
# Colab's preinstalled torchao 0.10 conflicts with PEFT and is not used here.
!pip uninstall -y torchao
!pip install -e .

In [ ]:
import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

drive_results = Path("/content/drive/MyDrive/Korean-TDCS/results")
local_results = Path.cwd() / "results"
drive_results.mkdir(parents=True, exist_ok=True)

if local_results.is_symlink():
    if local_results.resolve() != drive_results.resolve():
        raise RuntimeError(f"{local_results} points to the wrong Drive directory")
elif local_results.exists():
    shutil.copytree(local_results, drive_results, dirs_exist_ok=True)
    shutil.rmtree(local_results)

if not local_results.exists():
    local_results.symlink_to(drive_results, target_is_directory=True)

print(f"Saving all outputs to {drive_results}")

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

In [ ]:
# ---- User controls ----
TRAINING_BATCH_SIZE = 16  # GPU micro-batch. Reduce to 8 only if this OOMs.
EFFECTIVE_BATCH_SIZE = 32  # Must match the Random SFT baseline.
TRAINING_MAX_TOKENS = 3072  # Covers 99.94% of serialized examples.
TRAINING_EPOCHS = 4  # Full curriculum; no early stopping in Notebook 3.
EVAL_BATCH_SIZE = 16
EVAL_MAX_TOKENS = 4096  # Maximum newly generated tokens.
TDCS_RUN_DIR = Path("results/tdcs/olympiad_run_002")

if not 1 <= TRAINING_BATCH_SIZE <= EFFECTIVE_BATCH_SIZE:
    raise ValueError("TRAINING_BATCH_SIZE must be between 1 and EFFECTIVE_BATCH_SIZE")
if EFFECTIVE_BATCH_SIZE % TRAINING_BATCH_SIZE != 0:
    raise ValueError("EFFECTIVE_BATCH_SIZE must be divisible by TRAINING_BATCH_SIZE")
if TRAINING_EPOCHS < 1:
    raise ValueError("TRAINING_EPOCHS must be positive")

In [ ]:
from datasets import load_dataset
from korean_math_tdcs.utils.config import load_config

preflight_config = load_config("configs/tdcs.yaml")
expected_dataset = "Seungjun/Korean-Olympiad-TDCS-3K"
if preflight_config["data"]["dataset"] != expected_dataset:
    raise RuntimeError(f"Wrong training dataset: {preflight_config['data']['dataset']}")
if preflight_config["data"].get("format") != "olympiad_tdcs":
    raise RuntimeError("configs/tdcs.yaml is not configured for olympiad_tdcs")
raw_dataset = load_dataset(expected_dataset, token=userdata.get("HF_TOKEN"))
if len(raw_dataset["train"]) != 3000 or len(raw_dataset["validation"]) != 300:
    raise RuntimeError(
        f"Unexpected split sizes: train={len(raw_dataset['train'])}, "
        f"validation={len(raw_dataset['validation'])}"
    )
required_columns = {"problem_ko", "solution_ko", "difficulty_level"}
if not required_columns.issubset(raw_dataset["train"].column_names):
    raise RuntimeError("Training dataset schema is incorrect")
planned_steps = (len(raw_dataset["train"]) * TRAINING_EPOCHS + EFFECTIVE_BATCH_SIZE - 1) // EFFECTIVE_BATCH_SIZE
if planned_steps != 375:
    raise RuntimeError(f"Expected 375 optimizer steps, resolved {planned_steps}")
print("Preflight passed: correct dataset, 3,000/300 rows, 375 optimizer steps")

audit_cmd = ("python scripts/analyze_difficulty.py --config configs/tdcs.yaml "
             f"--set training.max_seq_length={TRAINING_MAX_TOKENS}")
!{audit_cmd}

train_cmd = ("python scripts/train_tdcs.py --config configs/tdcs.yaml "
             f"--set training.micro_batch_size={TRAINING_BATCH_SIZE} "
             f"--set training.effective_batch_size={EFFECTIVE_BATCH_SIZE} "
             f"--set training.max_seq_length={TRAINING_MAX_TOKENS} "
             f"--set training.epochs={TRAINING_EPOCHS} "
             f"--set output.run_dir={TDCS_RUN_DIR}")
!{train_cmd}

In [ ]:
BASELINE_RESULTS_PATH = (
    "results/baseline/olympiad_bench_math_ko/"
    f"max_tokens_{EVAL_MAX_TOKENS}/metrics.json"
)
SFT_RESULTS_PATH = Path("results/sft/olympiad_run_002") / f"evaluation_max_tokens_{EVAL_MAX_TOKENS}.json"
TDCS_RESULTS_PATH = TDCS_RUN_DIR / f"evaluation_max_tokens_{EVAL_MAX_TOKENS}.json"
TDCS_ADAPTER_PATH = TDCS_RUN_DIR / "adapter"
if not (TDCS_ADAPTER_PATH / "adapter_config.json").exists():
    raise FileNotFoundError("TDCS adapter missing; finish the training cell first")

cmd = ("python scripts/evaluate.py --config configs/baseline.yaml "
       f"--set evaluation.batch_size={EVAL_BATCH_SIZE} "
       f"--set evaluation.generation.max_new_tokens={EVAL_MAX_TOKENS} "
       f"--set model.adapter={TDCS_ADAPTER_PATH} "
       f"--set output.results_path={TDCS_RESULTS_PATH}")
!{cmd}

In [ ]:
import json
from pathlib import Path

import pandas as pd

tdcs = json.load(open(TDCS_RESULTS_PATH))
table = {"TDCS": {k: v["score"] for k, v in tdcs["benchmarks"].items()}}
for label, path in [("Random SFT", SFT_RESULTS_PATH), ("Base", Path(BASELINE_RESULTS_PATH))]:
    if path.exists():
        result = json.load(path.open())
        table[label] = {k: v["score"] for k, v in result["benchmarks"].items()}
pd.DataFrame(table)